# ライブ表示の試作

「ランタイム → すべてのセルを実行」で、各セルの進捗表示を確かめます。

In [ ]:
#@title 準備 { display-mode: "form" }

import os, sys, json, time, subprocess, importlib.util, re
from pathlib import Path
SOURCE_URL = "https://github.com/PenguinEino/tritflow.git"
SOURCE_REF = os.environ.get("TRITFLOW_REF", "0e5f92615d8bb1b751eafae4360431630ad9a53a")
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

def git(*args, progress=None):
    """Run git; with ``progress``, show git's own counter on one redrawn line."""
    if progress is None:
        return subprocess.check_output(["git", "-C", str(ROOT), *args], text=True).strip()
    from IPython.display import display, HTML
    style = "font:13px/1.6 ui-monospace,Menlo,Consolas,monospace;white-space:pre"
    line = display(HTML(f'<div style="{style}">⠿ {progress}</div>'), display_id=True)
    process, text, started = subprocess.Popen(["git", *args], stderr=subprocess.PIPE, text=True), "", time.monotonic()
    for chunk in iter(lambda: process.stderr.read(64), ""):
        text += chunk
        parts = [p.strip() for p in re.split(r"[\r\n]", text)[:-1] if p.strip()]
        if parts:
            line.update(HTML(f'<div style="{style}">⠿ <b>{progress}</b>  <span style="color:#8a8f98">{parts[-1][:80]}</span></div>'))
    if process.wait():
        raise RuntimeError(f"git {' '.join(args[:2])} failed:\n{text[-2000:]}")
    line.update(HTML(f'<div style="{style}"><span style="color:#1a9c4a">✓</span> <b>{progress}</b>  '
                     f'<span style="color:#8a8f98">{time.monotonic()-started:.1f}s</span></div>'))

if IN_COLAB:
    ROOT = Path("/content/tritflow")
    if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_REF):
        raise ValueError("TRITFLOW_REF must be a full 40-character commit")
    if not ROOT.exists():
        git("clone", "--progress", "--no-checkout", "--depth", "1", "--filter=blob:none",
            SOURCE_URL, str(ROOT), progress="GitHubからソースを取得")
        git("-C", str(ROOT), "fetch", "--progress", "--depth", "1", "origin", SOURCE_REF,
            progress=f"コミット {SOURCE_REF[:7]} を取得")
        git("-C", str(ROOT), "switch", "--detach", SOURCE_REF, progress="ファイルを展開")
    if git("rev-parse", "HEAD") != SOURCE_REF or git("status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError("/content/tritflow differs from the pinned commit; connect a fresh runtime")
else:
    candidates = [Path(os.environ.get("TRITFLOW_ROOT", ".")), Path.cwd(), Path.cwd().parent]
    ROOT = next((p.resolve() for p in candidates if (p/"tritflow").is_dir() and (p/"notebook").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("run from the TritFlow checkout or set TRITFLOW_ROOT")
sys.path[:0] = [str(ROOT), str(ROOT/"notebook")]
from live import step, run_stages
from runtime_support import SETUP_SCHEMA, setup_identity, check_runtime, print_tool_versions
runtime = ROOT/"notebook/runtime.json"
if IN_COLAB and (not runtime.is_file()
                 or json.loads(runtime.read_text()).get("TRITFLOW_SETUP_SHA256") != setup_identity(ROOT)):
    os.environ["TRITFLOW_NOTEBOOK_LANGUAGE"] = "ja"
    run_stages(["bash", str(ROOT/"notebook/colab_setup.sh"), "--source", str(ROOT)])
if runtime.is_file():
    os.environ.update(json.loads(runtime.read_text()))
    sys.path.insert(0, os.environ["TRITFLOW_PYTHON_TARGET"])
os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
with step("ツールを確認") as s:
    check_runtime()
    tools = print_tool_versions()
    s.done(" · ".join(re.search(r"(KLayout [\d.]+|ngspice-\d+)", v)[1] for v in tools.values()))
BUILD = Path(os.environ.get("TRITFLOW_NOTEBOOK_BUILD", "/content/build" if IN_COLAB else ROOT/"build/notebook")).resolve()

from live import need, provides, install_error_view, LayoutView
from tritflow import flow
install_error_view()
provides("design", "1-1 HDLを読み込む")
provides("net", "1-2 合成する")
provides("lay", "1-3 配置配線する")


In [ ]:
# 1-1 HDLを読み込む
source = """
module first(input trit a, output trit y);
  assign y = a & ~a;   // & is min
endmodule
"""
with step("HDLを読み込む") as s:
    design = flow.loads(source, "first")
    s.done("1入力 · 1出力")

In [ ]:
# 1-2 合成する
need("design", why="合成する回路のHDLは、1-1で読み込みます。")
with step("合成") as s:
    net = design.synthesize()
    s.done(" · ".join(f"{cell}×{n}" for cell, n in net.report()["cells"].items()))

In [ ]:
# 1-3 配置配線する
need("net", why="このセルは、合成した回路 net を配置配線します。")
with step("配置配線", view=LayoutView()) as s:
    lay = net.place_and_route(output_dir=BUILD/"first", seed=0)
    s.done(f"{lay.stats['instance_count']}セル · {lay.stats['die_size_um'][0]:.0f} × {lay.stats['die_size_um'][1]:.0f} µm")

In [ ]:
# 1-3b 2トリット加算器でも配置配線する（大きめの例）
with step("配置配線（2トリット加算器）", view=LayoutView()) as s:
    adder = flow.load(ROOT/"examples/adder2.tv", "adder2").synthesize()
    adder_lay = adder.place_and_route(output_dir=BUILD/"adder2", seed=0)
    s.done(f"{adder_lay.stats['instance_count']}セル · {adder_lay.stats['die_size_um'][0]:.0f} × {adder_lay.stats['die_size_um'][1]:.0f} µm")

In [ ]:
# 1-4 DRCとLVS
need("lay", why="DRCとLVSは、1-3で作ったGDSを調べます。")
with step("DRC") as s:
    drc = lay.drc()
    s.done(f"違反 {drc['violations']}件")
with step("LVS") as s:
    lvs = lay.lvs()
    s.done("一致" if lvs["matched"] else "不一致")

In [ ]:
# 1-5 抽出後のSPICE
need("lay", why="SPICEは、1-3で作ったGDSから取り出した回路を動かします。")
vectors = [(-1,), (0,), (1,), (0,), (-1,)]
with step("抽出後SPICE") as s:
    post = lay.extract().simulate(vectors, load=100e-15, expected=lambda v: {"y": min(v[0], -v[0])},
                                  hold_s=1e-6, step_s=2e-9, edge_s=1e-9, temperature_c=27)
    s.done("全区間で合格" if post["measurements"]["passed"] else "不合格")

In [ ]:
# 1-6 振り返る
need("lay", why="配置配線の途中経過は、1-3の実行で記録されます。")
from tritflow.viz.layout import placement_animation, routing_progress
placement_animation(lay.data).show()
routing_progress(lay.data).show()

In [ ]:
# 順番を間違えた例：未定義の変数を使う
del lay
with step("DRC") as s:
    lay.drc()